In [1]:
from pyspark.sql import SparkSession

try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("Real-Time Lakehouse") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.demo", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.demo.type", "nessie") \
    .config("spark.sql.catalog.demo.uri", "http://nessie:19120/api/v1") \
    .config("spark.sql.catalog.demo.ref", "main") \
    .config("spark.sql.catalog.demo.authentication.type", "NONE") \
    .config("spark.sql.catalog.demo.warehouse", "s3a://warehouse") \
    .config("spark.sql.catalog.demo.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.demo.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.demo.s3.path-style-access", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("✅ Đã khởi tạo Spark Session thành công!")

✅ Đã khởi tạo Spark Session thành công!


26/05/21 02:49:28 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
!pip install boto3 --quiet


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:

import boto3
from botocore.client import Config

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="admin",
    aws_secret_access_key="password",
    config=Config(signature_version="s3v4"),
)

# Tạo bucket nếu chưa có
buckets = [b["Name"] for b in s3.list_buckets()["Buckets"]]
if "warehouse" not in buckets:
    s3.create_bucket(Bucket="warehouse")
    print("✅ Đã tạo bucket 'warehouse'!")
else:
    print("✅ Bucket 'warehouse' đã tồn tại!")

✅ Bucket 'warehouse' đã tồn tại!


In [2]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import from_json, col

# Stop tất cả query đang chạy trước khi tạo mới
for q in spark.streams.active:
    q.stop()
    print(f"⏹ Đã stop: {q.id}")
# 1. Định nghĩa cấu trúc dữ liệu (Schema) cho bảng Customers
# Dựa trên bộ dữ liệu Olist Customers
schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_unique_id", StringType(), True),
    StructField("customer_zip_code_prefix", IntegerType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True)
])

# 2. Đọc luồng dữ liệu từ Kafka (Topic: raw_customers)
print("Đang kết nối với Kafka...")
kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "raw_customers") \
    .option("startingOffsets", "earliest") \
    .load()

# 3. Parse dữ liệu JSON từ Kafka
# Kafka lưu dữ liệu dưới dạng binary (mặc định là cột 'value'), ta cần ép kiểu về chuỗi và parse JSON
parsed_stream = kafka_stream \
    .selectExpr("CAST(value AS STRING) as json_str") \
    .select(from_json(col("json_str"), schema).alias("data")) \
    .select("data.*")

# 4. Chuẩn bị Tầng Bronze trên Iceberg
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.bronze")

# 5. Ghi luồng dữ liệu thẳng vào bảng Iceberg
print("Bắt đầu ghi dữ liệu streaming vào Iceberg (Tầng Bronze)...")
query = parsed_stream.writeStream \
    .format("iceberg") \
    .outputMode("append") \
    .option("checkpointLocation", "/home/iceberg/warehouse/checkpoints/bronze_customers") \
    .trigger(processingTime="30 seconds") \
    .queryName("bronze_customers") \
    .toTable("demo.bronze.customers")

print("Pipeline Streaming đang chạy! Dữ liệu đang được nạp liên tục...")

Đang kết nối với Kafka...
Bắt đầu ghi dữ liệu streaming vào Iceberg (Tầng Bronze)...
Pipeline Streaming đang chạy! Dữ liệu đang được nạp liên tục...


In [3]:
# Theo dõi số dòng tăng dần để xác nhận pipeline đang hoạt động
import time

for i in range(20):  # chạy 20 lần × 3s = theo dõi 1 phút
    count = spark.sql("SELECT COUNT(*) FROM demo.bronze.customers").collect()[0][0]
    active = len(spark.streams.active)
    print(f"[{i+1}] Dòng trong Bronze: {count} | Query active: {active}")
    time.sleep(30)

[1] Dòng trong Bronze: 590 | Query active: 1
[2] Dòng trong Bronze: 1818 | Query active: 1
[3] Dòng trong Bronze: 4586 | Query active: 1
[4] Dòng trong Bronze: 7368 | Query active: 1
[5] Dòng trong Bronze: 10130 | Query active: 1
[6] Dòng trong Bronze: 12898 | Query active: 1
[7] Dòng trong Bronze: 15662 | Query active: 1
[8] Dòng trong Bronze: 18433 | Query active: 1
[9] Dòng trong Bronze: 21209 | Query active: 1
[10] Dòng trong Bronze: 23977 | Query active: 1
[11] Dòng trong Bronze: 26749 | Query active: 1
[12] Dòng trong Bronze: 29516 | Query active: 1
[13] Dòng trong Bronze: 32294 | Query active: 1
[14] Dòng trong Bronze: 35065 | Query active: 1
[15] Dòng trong Bronze: 37846 | Query active: 1
[16] Dòng trong Bronze: 40579 | Query active: 1
[17] Dòng trong Bronze: 43301 | Query active: 1
[18] Dòng trong Bronze: 46013 | Query active: 1
[19] Dòng trong Bronze: 48754 | Query active: 1
[20] Dòng trong Bronze: 51477 | Query active: 1


In [4]:
df = spark.sql("SELECT * FROM demo.bronze.customers LIMIT 10")
df.show()
print(f"Tổng số dòng: {spark.sql('SELECT COUNT(*) FROM demo.bronze.customers').collect()[0][0]}")

+--------------------+--------------------+------------------------+---------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|  customer_city|customer_state|
+--------------------+--------------------+------------------------+---------------+--------------+
|dd32ce6c2b55f4746...|26637eea804123401...|                   13010|       campinas|            SP|
|49cbac2b39ac1280d...|a9e4348b76f85861a...|                   78740|   rondonopolis|            MT|
|ac21bf9bfae1fe0cd...|b9b4ef29650e6efa5...|                    5885|      sao paulo|            SP|
|91d98668d81671694...|7bd43abfde2d6a9a5...|                    4315|      sao paulo|            SP|
|44f3a50c04eb44a44...|8b4d7eb805eedb498...|                   29102|     vila velha|            ES|
|96fd44add323bdcc8...|bd204d0a603fa4506...|                   26250|    nova iguacu|            RJ|
|45360c874acf24110...|b703d83e085a6a8b3...|                   25070|duque de caxias|            RJ|


In [5]:
print(f"Số query đang chạy: {len(spark.streams.active)}")
for q in spark.streams.active:
    print(f"  - Query: {q.name}, Status: {q.status}")

Số query đang chạy: 1
  - Query: bronze_customers, Status: {'message': 'Waiting for next trigger', 'isDataAvailable': False, 'isTriggerActive': False}


In [6]:
import boto3
from botocore.client import Config

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="admin",
    aws_secret_access_key="password",
    config=Config(signature_version="s3v4"),
)

objects = s3.list_objects_v2(Bucket="warehouse", Prefix="bronze/")
count = objects.get("KeyCount", 0)
print(f"Số file trong bucket warehouse/bronze: {count}")
if count > 0:
    for obj in objects["Contents"][:5]:
        print(f"  - {obj['Key']} ({obj['Size']} bytes)")

Số file trong bucket warehouse/bronze: 306
  - bronze/customers_6172231c-6d4b-46e0-9480-dc93d427dd33/data/00000-0-d3d1e87b-4662-4211-9b76-19490360fcb1-45-00001.parquet (108919 bytes)
  - bronze/customers_6172231c-6d4b-46e0-9480-dc93d427dd33/data/00000-11-d3d1e87b-4662-4211-9b76-19490360fcb1-46-00001.parquet (86035 bytes)
  - bronze/customers_6172231c-6d4b-46e0-9480-dc93d427dd33/data/00000-120-d3d1e87b-4662-4211-9b76-19490360fcb1-53-00001.parquet (118187 bytes)
  - bronze/customers_6172231c-6d4b-46e0-9480-dc93d427dd33/data/00000-134-d3d1e87b-4662-4211-9b76-19490360fcb1-54-00001.parquet (117282 bytes)
  - bronze/customers_6172231c-6d4b-46e0-9480-dc93d427dd33/data/00000-155-d3d1e87b-4662-4211-9b76-19490360fcb1-55-00001.parquet (118238 bytes)
